# Retrieval, ablation and router evaluation

**Owner:** Aliyan (retrieval/ablation) and Sohaib (router)

Skeleton only: fill in the empty code cells; do not change contracts.

In [1]:
# Data paths: run this cell first.
from pathlib import Path

REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_DIR / "data" / "papers.parquet"
EMBEDDINGS_PATH = REPO_DIR / "data" / "embeddings.npy"
DATA_PATH, EMBEDDINGS_PATH


(WindowsPath('D:/Sem_7/DS_Project/research_paper_discovery/data/papers.parquet'),
 WindowsPath('D:/Sem_7/DS_Project/research_paper_discovery/data/embeddings.npy'))

## Load eval files

Read eval/eval_retrieval.json and eval/eval_routing.json.

In [ ]:
import json

retrieval_eval = json.loads((REPO_DIR / "eval" / "eval_retrieval.json").read_text(encoding="utf-8"))
routing_eval = json.loads((REPO_DIR / "eval" / "eval_routing.json").read_text(encoding="utf-8"))
print({"retrieval_items": len(retrieval_eval), "routing_items": len(routing_eval)})

## Score retrieval

Call hit_rate_at_k and mrr on the hand-labelled ids.

In [ ]:
from src.evaluate import hit_rate_at_k, mrr
from src.rag import retrieve

results = []
gold = []
for item in retrieval_eval:
    filters = item["filters"]
    docs = retrieve(
        item["question"], k=5,
        year_min=filters.get("year_min"), year_max=filters.get("year_max"),
        category=filters.get("category"), topic=filters.get("topic"),
    )
    results.append([doc["arxiv_id"] for doc in docs])
    gold.append(list(item.get("relevant_ids", [])))

print({
    "hit_rate_at_5": hit_rate_at_k(results, gold, 5) if any(gold) else "PRELIMINARY: no verified labels",
    "mrr": mrr(results, gold) if any(gold) else "PRELIMINARY: no verified labels",
})

## Run the ablation

Call src.evaluate.run_ablation on the eval questions.

In [ ]:
from src.evaluate import run_ablation

ablation = run_ablation([item["question"] for item in retrieval_eval])
ablation

## Citation review sample

Call src.evaluate.citation_sample_csv for manual checking.

In [ ]:
from src.evaluate import citation_sample_csv

citation_sample_csv(n=20, path=REPO_DIR / "eval" / "citation_sample.csv")

## Router accuracy (Sohaib)

Call src.evaluate.router_accuracy on the routing gold set.

In [2]:
import json, sys
sys.path.insert(0, str(REPO_DIR))
from src import nodes, evaluate
import pandas as pd

routing = json.load(open(REPO_DIR/"eval"/"eval_routing.json", encoding="utf-8"))
questions = [r["question"] for r in routing]
gold = [r["route"] for r in routing]

preds = []
for q in questions:
    st = nodes.router_node({"question": q, "llm_calls": 0})
    preds.append(st["route"])

ra = evaluate.router_accuracy(preds, gold)
print("router accuracy:", round(ra["accuracy"], 4))
print("n =", len(gold))

cm = ra["confusion"]
routes = list(cm.keys())
df_cm = pd.DataFrame(cm, index=routes, columns=routes).fillna(0).astype(int)
print(df_cm.to_string())

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(5,4))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Gold")
plt.title("Router confusion matrix")
plt.tight_layout()
plt.savefig(REPO_DIR/"images"/"10_router_confusion.png", dpi=120)
plt.close()
print("saved image")


STUB: llm_json


STUB: llm_json


STUB: llm_json


STUB: llm_json


STUB: llm_json


STUB: llm_json


... (32 outputs left out)

STUB: llm_json


STUB: llm_json


router accuracy: 0.8
n = 40
            literature  trend  hybrid  fallback
literature          10      1       0         1
trend                0      3       0         0
hybrid               0      5      10         0
fallback             0      1       0         9


saved image


## Results

TODO: summarize the outputs and figures produced above.